In [1]:
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache

In [2]:
from langchain_groq import ChatGroq

c:\Users\Krishna\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
from dotenv import load_dotenv

In [5]:
intent_llm = ChatGroq(model="openai/gpt-oss-120b",api_key=os.environ.get("GROQ_API_KEY"))
component_llm = ChatGroq(model="openai/gpt-oss-120b", api_key=os.environ.get("GROQ_API_KEY"))

In [6]:
intent_llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F8FBE839D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F8FBE838E0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
from typing import TypedDict, Annotated, List, Optional
from pydantic import BaseModel, Field
import re
import json as jsonlib

In [8]:
from langgraph.graph.message import add_messages
from langchain_core.prompts import ChatPromptTemplate

In [9]:
class UserState(BaseModel):
    budget: Optional[int] = None
    family_size: Optional[int] = None
    fuel_preference:Optional[str] = None
    usage_pattern:Optional[str] = None
    location:Optional[str] = None
    needs_interruption: bool = False
    query_type: str = 'general'

In [10]:
class AgentOutput(BaseModel):
    text: str
    component: Optional[dict] = None
    follow_up: str | None

In [11]:
class GraphState(TypedDict):
    messages: Annotated[list, add_messages]
    user_state: UserState
    last_agent_output: Optional[AgentOutput]

In [12]:
def classify_user_intent(state: GraphState):
    messages = state['messages'][-1].content
    user_state = state['user_state']

    intent_prompt = ChatPromptTemplate.from_template("""
You are analyzing a user message in a Tata Motors car sales conversation.
User Message: {messages}
Current Profile: {user_state}

Your job is to update the user profile based ONLY on what is explicitly stated in the message.

Output ONLY a raw JSON object — no markdown, no ```json, no explanation.
First character must be {{ and last must be }}

{{
  "budget": <integer in rupees or null>,
  "family_size": <integer or null>,
  "fuel_preference": <"petrol" | "diesel" | "cng" | "electric" | null>,
  "usage_pattern": <"city" | "highway" | "mixed" | null>,
  "location": <city name as string or null>,
  "needs_interruption": <true if user wants Test Drive or Booking, else false>,
  "query_type": <"general" | "comparison_table" | "car_card" | "spec_table" | "show_calculation">
}}

Rules:
- Do NOT assume anything not explicitly stated. If unsure, keep existing value or null.
- needs_interruption → true only if user explicitly says "test drive", "booking", "interested in buying", "ready to buy", "want to purchase", etc.
- query_type rules:
  * "comparison_table" → comparing 2 or more cars: "Nexon vs Safari", "which is better Punch or Altroz"
  * "car_card"         → asking about one specific car:  "tell me about Harrier", "what about Safari"
  * "spec_table"       → specific specs of one car: "Nexon's mileage", "Safari boot space", "Harrier engine"
  * "show_calculation" → any calculation: "Calculate EMI", "on-road price", "trade in value"
  * "general"          → everything else: greetings, vague questions, recommendations, "which car should I buy", "best car for me","im poor ass which car to buy" etc.
""")

    chain = intent_prompt | intent_llm
    intent = chain.invoke({"messages":messages, "user_state":user_state}).content

    updated_state = UserState.model_validate_json(intent)
    return {"user_state": updated_state}

In [13]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma


In [14]:
embeddings = OllamaEmbeddings(model="llama3.2:3b")

C:\Users\Krishna\AppData\Local\Temp\ipykernel_2108\1727108800.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="llama3.2:3b")


In [15]:
vectorDb = Chroma(
    persist_directory='./tata_motors_knowledge_base',
    embedding_function=embeddings
)

In [16]:
retriver = vectorDb.as_retriever()

In [17]:
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

In [18]:
from operator import itemgetter


In [19]:
comparison_table = """
For COMPARISON queries (Nexon vs Safari, Harrier vs Safari, etc.):
{{
  "text": "<brief natural summary of comparison  - if any numbers must be described in their numerical format and not in their word format>",
  "component": {{
    "required": true,
    "name": "comparison_table",
    "component_script": "<natural language summary of the contents of the table >",
    "content": {{
      "columns": [
        {{ "key": "feature", "label": "Feature" }},
        {{ "key": "car1", "label": "<Car 1 Name>" }},
        {{ "key": "car2", "label": "<Car 2 Name>" }}
      ],
      "rows": [
        {{ "feature": "<feature name>", "car1": "<value>", "car2": "<value>" }}
      ]
    }}
  }},
  "follow_up": "<question guiding towards purchase>"
}}
"""

In [20]:
car_card = """
For SINGLE CAR info queries:
{{
  "text": "<natural language summary >",
  "component": {{
    "required": true,
    "name": "car_card",
    "content": {{
      "model": "<exact car model name>"
    }}
  }},
  "follow_up": "<question guiding towards purchase>"
}}
"""

In [21]:
spec_table = """
For SPECIFICATION queries (one car's specs):
{{
  "text": "<natural language summary>",
  "component": {{
    "required": true,
    "name": "spec_table",
    "component_script": "<natural language summary of the contents of the table>",
    "content": {{
      "columns": [
        {{ "key": "feature", "label": "Feature" }},
        {{ "key": "value", "label": "Details" }}
      ],
      "rows": [
        {{ "feature": "<name>", "value": "<value>" }}
      ]
    }}
  }},
  "follow_up": "<question guiding towards purchase>"
}}
"""

In [22]:
general = """
For GENERAL queries (no component needed):
{{
  "text": "<natural language answer>",
  "component": {{
    "required": false,
    "name": null,
    "content": null
  }},
  "follow_up": "<question guiding towards purchase>"
}}
"""

In [23]:
show_calculation = """
FOR CALCULATION queries (e.g., EMI calculation):
{{
"text": "<brief natural language summary of calculation>",
"component":
{{
  "required": true,
  "name": "show_calculation",
  "component_script": "<natural language summary of the calculation steps and result IN THE MOST SIMPLE WORDS POSSIBLE>",
  "content": {{
    "title": "<name of the calculation>",
    "inputs": [
      {{
        "label": "<input name>",
        "value": "<input value>"
      }}
    ],
    "steps": [
      {{
        "step": "<calculation step description>",
        "result": "<intermediate result>"
      }}
    ],
    "result": {{
      "label": "<final output name>",
      "value": "<final value>"
    }}
  }}
}}
"""

In [24]:
examples = """
*EXAMPLES TO LEARN FROM:**
Example - Comparison (Realistic):
Q: "I'm confused between Nexon and Safari. Which one should I go for if I have a family of 5?"
A: {{"text": "Nexon is a compact SUV (₹7-14L, 5 seats) while Safari is a full-size SUV (₹15-25L, 7 seats).", "component": {{"required": true, "name": "comparison_table", "component_script": "Nexon is a smaller 5-seater SUV with a lower price range, while Safari is a bigger 6 or 7-seater SUV with a higher price range.", "content": {{"columns": [{{"key": "feature", "label": "Feature"}}, {{"key": "car1", "label": "Tata Nexon"}}, {{"key": "car2", "label": "Tata Safari"}}], "rows": [{{"feature": "Seating", "car1": "5", "car2": "6 or 7"}}, {{"feature": "Price", "car1": "₹7-14L", "car2": "₹15-25L"}}]}}}}, "follow_up": "Do you need extra seating often or mainly city driving?"}}

Example - Single Car (Exploration):
Q: "I've been hearing a lot about the Tata Nexon lately. What makes it so popular?"
A: {{"text": "Tata Nexon is a compact SUV for young professionals with ₹7-14L budget and 5-star safety.", "component": {{"required": true, "name": "car_card", "content": {{"model": "Tata Nexon"}}}}, "follow_up": "Would you like to explore variants or compare with competitors?"}}

Example - Specification (Practical):
Q: "Can you tell me the key specifications of the Tata Punch? I want something for daily city use."
A: {{"text": "Tata Punch is a compact micro SUV designed for city driving with good ground clearance and efficiency.", "component": {{"required": true, "name": "spec_table", "component_script": "Tata Punch comes with a petrol engine, offers good mileage, and has high ground clearance suitable for city roads.", "content": {{"columns": [{{"key": "feature", "label": "Feature"}}, {{"key": "value", "label": "Details"}}], "rows": [{{"feature": "Engine", "value": "1.2L Petrol"}}, {{"feature": "Mileage", "value": "20 kmpl"}}, {{"feature": "Ground Clearance", "value": "187 mm"}}]}}}}, "follow_up": "Is your usage mostly city driving or occasional highway trips as well?"}}

Example - General (Decision):
Q: "I drive mostly in the city and occasionally on highways. What kind of car should I be looking at?"
A: {{"text": "For city driving with occasional highway use, compact SUVs like Nexon or Punch are great options as they offer comfort, mileage, and good ground clearance.", "component": {{"required": false, "name": null, "content": null}}, "follow_up": "What is your budget range so I can suggest the best option?"}}

Example - Comparison (Feature Focus):
Q: "Between Harrier and Safari, is there any real difference apart from seating?"
A: {{"text": "Harrier is a 5-seater while Safari offers 6 or 7 seats, with similar engine and features otherwise.", "component": {{"required": true, "name": "comparison_table", "component_script": "Both Harrier and Safari share similar engine and features, but Safari offers an extra row of seating compared to Harrier.", "content": {{"columns": [{{"key": "feature", "label": "Feature"}}, {{"key": "car1", "label": "Tata Harrier"}}, {{"key": "car2", "label": "Tata Safari"}}], "rows": [{{"feature": "Seating", "car1": "5", "car2": "6 or 7"}}, {{"feature": "Engine", "car1": "Same", "car2": "Same"}}, {{"feature": "3rd Row", "car1": "No", "car2": "Yes"}}]}}}}, "follow_up": "Do you actually need the third row regularly?"}}

Example - General (Early Stage):
Q: "I'm planning to buy a car in the next few months but haven't decided anything yet. Where should I start?"
A: {{"text": "A good starting point is deciding your budget, usage type, and how many people you usually travel with.", "component": {{"required": false, "name": null, "content": null}}, "follow_up": "What is your rough budget and primary usage?"}}
"""

In [25]:
calculation_examples = """
Example - EMI Calculation:
Q: "If I buy a car for ₹11.5 lakh and pay ₹2.5 lakh upfront, what will my EMI be for 5 years?"
A: {{"text": "Here’s the EMI calculation based on your inputs:", "component": {{"required": true, "name": "show_calculation", "component_script": "The car price is 11 lakh 50 thousand. You pay 2 lakh 50 thousand upfront. The remaining loan is 9 lakh. At 9.5 percent interest for 5 years, the EMI is about 18 thousand 900 per month.", "content": {{"title": "Car EMI Calculation", "inputs": [{{"label": "On-road Price", "value": "₹11,50,000"}}, {{"label": "Down Payment", "value": "₹2,50,000"}}, {{"label": "Loan Amount", "value": "₹9,00,000"}}, {{"label": "Interest Rate", "value": "9.5%"}}, {{"label": "Tenure", "value": "60 months"}}], "steps": [{{"step": "Loan Amount = Price - Down Payment", "result": "₹9,00,000"}}, {{"step": "Monthly Interest Rate = 9.5 / 12", "result": "0.7917%"}}, {{"step": "EMI Calculation", "result": "₹18,900 per month"}}], "result": {{"label": "Monthly EMI", "value": "₹18,900"}}}}}, "follow_up": "Do you want to check EMI for a different down payment?"}}
'
Example - On-road Cost:
Q: "If a car’s ex-showroom price is ₹10.8 lakh, what would be the total on-road price?"
A: {{"text": "Here’s the estimated on-road cost breakdown:", "component": {{"required": true, "name": "show_calculation", "component_script": "The ex-showroom price is 10 lakh 80 thousand. Road tax is about 3 percent, which is around 32 thousand. Insurance is about 40 thousand, and other charges are around 20 thousand. So the total on-road price comes to around 11 lakh 72 thousand.", "content": {{"title": "On-road Price Calculation", "inputs": [{{"label": "Ex-showroom Price", "value": "₹10,80,000"}}], "steps": [{{"step": "Road Tax (3%)", "result": "₹32,400"}}, {{"step": "Insurance", "result": "₹40,000"}}, {{"step": "Other Charges", "result": "₹20,000"}}], "result": {{"label": "Total On-road Price", "value": "₹11,72,400"}}}}}, "follow_up": "Do you want EMI calculation on this on-road price?"}}

Example - Fuel Cost:
Q: "If I drive around 12,000 km every year, how much will I spend on fuel in 5 years for a petrol car?"
A: {{"text": "Here’s the total fuel cost over 5 years:", "component": {{"required": true, "name": "show_calculation", "component_script": "You drive 12 thousand kilometers per year, so in 5 years that is 60 thousand kilometers. If mileage is 18 kilometers per liter, you need about 3333 liters of fuel. At 100 rupees per liter, the total cost becomes around 3 lakh 33 thousand.", "content": {{"title": "Fuel Cost Calculation (5 Years)", "inputs": [{{"label": "Annual Distance", "value": "12,000 km"}}, {{"label": "Mileage", "value": "18 km/l"}}, {{"label": "Fuel Price", "value": "₹100/l"}}, {{"label": "Duration", "value": "5 years"}}], "steps": [{{"step": "Total Distance in 5 years", "result": "60,000 km"}}, {{"step": "Fuel Required", "result": "3333 liters"}}, {{"step": "Total Fuel Cost", "result": "₹3,33,300"}}], "result": {{"label": "5-Year Fuel Cost", "value": "₹3,33,300"}}}}}, "follow_up": "Do you want to compare this with diesel cost?"}}

Example - Loan Eligibility:
Q: "My monthly salary is ₹60,000. How much EMI can I comfortably afford?"
A: {{"text": "Here’s your estimated EMI affordability:", "component": {{"required": true, "name": "show_calculation", "component_script": "Your monthly salary is 60 thousand. Ideally, EMI should be around 30 percent, which is 18 thousand. So you can comfortably afford an EMI of around 18 thousand per month.", "content": {{"title": "EMI Affordability", "inputs": [{{"label": "Monthly Salary", "value": "₹60,000"}}], "steps": [{{"step": "30% of Salary", "result": "₹18,000"}}, {{"step": "Recommended EMI Range", "result": "₹15,000 - ₹18,000"}}], "result": {{"label": "Affordable EMI", "value": "₹18,000"}}}}}, "follow_up": "Do you want to see which cars fit this EMI?"}}

Example - Down Payment Impact:
Q: "If I increase my down payment from ₹2 lakh to ₹3 lakh, how much will my EMI reduce?"
A: {{"text": "Here’s how your EMI changes with higher down payment:", "component": {{"required": true, "name": "show_calculation", "component_script": "With 2 lakh down payment, the loan is 9 lakh and EMI is about 18 thousand 900. With 3 lakh down payment, the loan reduces to 8 lakh and EMI becomes about 16 thousand 800. So your EMI reduces by around 2 thousand 100.", "content": {{"title": "Down Payment Impact", "inputs": [{{"label": "Car Price", "value": "₹11,00,000"}}, {{"label": "Old Down Payment", "value": "₹2,00,000"}}, {{"label": "New Down Payment", "value": "₹3,00,000"}}], "steps": [{{"step": "Old EMI (Loan ₹9L)", "result": "₹18,900"}}, {{"step": "New EMI (Loan ₹8L)", "result": "₹16,800"}}, {{"step": "EMI Reduction", "result": "₹2,100"}}], "result": {{"label": "EMI Difference", "value": "₹2,100 less"}}}}}, "follow_up": "Do you want to optimize EMI further or keep more cash in hand?"}}

Example - Warranty Break-even:
Q: "If the extended warranty costs ₹30,000, is it actually worth taking over 5 years?"
A: {{"text": "Here’s a break-even analysis for the extended warranty:", "component": {{"required": true, "name": "show_calculation", "component_script": "The warranty costs 30 thousand. Regular service over 5 years is around 50 thousand. If you face even one major repair costing 20 thousand or more, the warranty becomes useful. So if repair costs go above 30 thousand, you recover the warranty cost.", "content": {{"title": "Warranty Break-even Analysis", "inputs": [{{"label": "Warranty Cost", "value": "₹30,000"}}, {{"label": "5-Year Service Cost", "value": "₹50,000"}}, {{"label": "Possible Major Repair", "value": "₹20,000+"}}], "steps": [{{"step": "Total Warranty Cost", "result": "₹30,000"}}, {{"step": "Regular Service Cost (5 years)", "result": "₹50,000"}}, {{"step": "Break-even Point", "result": "Repair cost ≥ ₹30,000"}}], "result": {{"label": "Is Warranty Worth It?", "value": "Yes, if major repairs occur"}}}}}, "follow_up": "How many kilometers do you usually drive per year?"}}

"""

In [26]:
from langchain_core.messages import AIMessage, HumanMessage

In [27]:
def build_history(messages):
    history = ""
    for message in messages:
        if isinstance(message,HumanMessage):
             history += f"User Said: {message.content}"
        else:
            history += message.content
           
        
    return history

In [28]:
def retriver_and_component(state:GraphState):
    user_state = state['user_state']
    message = state['messages'][-1].content
    history = build_history(state['messages'])
    docs = retriver.invoke(message)
    context = '\n\n'.join(d.page_content for d in docs)
    if len(docs) == 0:
        context = ""
    prompt = ChatPromptTemplate.from_template("""
        User Query: {message}                                     
        The user has the following profile: {user_state}  
        You have to answer from context IF AND ONLY IF it is NON-EMPTY, provided below \n: {context}     
        Else, use your own knowledge  to answer the query.
        Please also refer the past history to answer the query.                               
        You are an AI sales assistant working for TATA Motors\n
        Here are your CORE responsibilites as far tone is concerned must revolve around:\n                                                                       
        1. Selling cars,
        2. Neogiating - *NOT* *Related* to prices - You must negotiate like a real car dealer - YOU MUST SOMEHOW CONVINCE PERSON TO BUY THE CAR,
        3. Upselling opportunities,
        4. Help user showing :\n
        a. EMI options & offers by elaborating the calculations step-by-step 
        b. RTO Calculations - by elaborating the calculations step-by-step
        c. Trade-in value calculations - by elaborating the calculations step-by-step                                      
                                                                                    
        However, you have to Output only a JSON object,for various types of query:
        Component: {component}
        Here are some examples to learn from: {examples}
        For Calculation related queries, here are some more detailed examples to learn from: {calculation_examples}      
        Chat History : {history}
        ADDITIONAL TIPS:
        SEEM VERY CONVINCING YET NOT VERY PUSHY. BE REALLY VERY SOFT.
        WHEREVER NUMBERS ARE THERE. PLACE IT IN THE NUMERICAL FORM THAN JUST WORD FORMAT.
        RETURN ONLY A VALID JSON OBJECT ONLY, FOLLOW THE ABOVE FORMAT VERY STRICTLY, "NO" MARKDOWNS            
                                                                                                                                 
""")
    chain = prompt | component_llm
    response = chain.invoke({"message":message, "user_state":user_state,"context":context,"component":user_state.query_type,"examples":examples,"calculation_examples":calculation_examples,"history":history}).content
    updated_state = AgentOutput.model_validate_json(response)
    return {"last_agent_output": updated_state, "messages": [AIMessage(content=f"ASSISTANT SAID: {updated_state.text} | ASSISTANT ASKED: {updated_state.follow_up}  ")]}

In [29]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver


In [30]:
memory = MemorySaver()

In [31]:
workflow = StateGraph(GraphState)
workflow.add_node("intent_node",classify_user_intent)
workflow.add_node("retriever_component_node", retriver_and_component)
workflow.add_edge(START,"intent_node")
workflow.add_edge("intent_node","retriever_component_node")
workflow.add_edge("retriever_component_node",END)

In [32]:
graph = workflow.compile(checkpointer=memory)

In [33]:
config = {"configurable": {"thread_id": "user_session_xyz"}}  # Example of a configurable parameter

In [42]:

state: GraphState = {
    "messages": ["around 15L"],  # Starting message from user
    "user_state": UserState(),
    "last_agent_output": None
}


In [43]:
response = graph.invoke(state,config=config)

In [36]:
current_state = graph.get_state(config)
print(current_state.values["messages"])


[HumanMessage(content='explain me tata harrier', additional_kwargs={}, response_metadata={}, id='86016a46-f84c-4796-995b-acda0c53b216'), AIMessage(content='ASSISTANT SAID: The Tata Harrier is a premium 5‑seat SUV that blends bold design with powerful performance. It’s powered by a 2.0\u202fL diesel engine delivering 350\u202fNm of torque, giving you strong pull‑away and confident highway cruising. Pricing ranges from ₹14.00\u202fL to ₹25.25\u202fL depending on the variant and optional extras. Key highlights include:\n- Spacious boot of 382\u202fL and a generous 200\u202fmm ground clearance for comfortable city and off‑road driving.\n- Premium comforts such as a panoramic sunroof, leather‑upholstered seats, and a 10.25‑inch touchscreen infotainment system.\n- Advanced safety with dual‑airbags, ABS with EBD, and a robust body structure.\n- Fuel efficiency of around 16.8\u202fkm/l (ARAI) for the diesel engine.\nYou’ll also enjoy optional up‑charges like dual‑tone paint (₹10‑15\u202fk) and

In [44]:
response['last_agent_output']

AgentOutput(text='Based on your budget of around ₹15\u202fL, the Tata Harrier XM at ₹15,20,000 is a great fit. It offers the 2.0\u202fL diesel engine, premium features like a 10.25‑inch touchscreen, and a comfortable 5‑seat cabin.', component={'required': True, 'name': 'car_card', 'content': {'model': 'Tata Harrier', 'variant': 'XM', 'price': '₹15,20,000'}}, follow_up='Would you like to see a personalized EMI plan based on a down payment you have in mind?')